# SRUL Colab quickstart

This notebook checks the environment, runs a minimal validation job, and provides the commands for the CIFAR-10 experiments. Clone or upload the repository before running the cells.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip install -q -r requirements.txt


## Minimal validation run

This one-epoch run uses generated images to verify the autoencoder, chord baseline, RFM prior, sampling loop, and checkpoint output.


In [ ]:
!python src/srul_cifar_spatial_tokens_experiment.py \
  --dataset fake \
  --out-dir runs/smoke \
  --train-samples 256 \
  --test-samples 128 \
  --ae-epochs 1 \
  --prior-epochs 1 \
  --base-channels 16 \
  --latent-channels 8 \
  --prior-width 32 \
  --prior-depth 2 \
  --batch-size 32 \
  --methods chord rfm \
  --sample-steps 4 \
  --metric-samples 64 \
  --pr-samples 64 \
  --recon-metric-samples 64 \
  --skip-heavy-metrics


## Main CIFAR-10 geometry experiment

The following cell reproduces the controlled comparison used in the report. It can take several hours on a Colab GPU. Checkpoints are resumable.


In [ ]:
!bash scripts/run_cifar10_geometry.sh runs


## Conditional model and classifier-free guidance

After the geometry run finishes, pass its autoencoder checkpoint to the conditional launcher.


In [ ]:
AE = "runs/cifar10_geometry/seed_0/checkpoints/autoencoder_final.pt"
!bash scripts/run_cifar10_conditional.sh runs "$AE"


## Information-control sweep


In [ ]:
AE = "runs/cifar10_geometry/seed_0/checkpoints/autoencoder_final.pt"
!bash scripts/run_cifar10_sigma_sweep.sh runs "$AE"


## Baseline and VAE checkpoint preparation

This run evaluates the matched reference systems and produces the VAE checkpoint and latent statistics needed by the next experiment.


In [ ]:
# Set the completed spherical checkpoints before running.
%env SRUL_AE_CHECKPOINT=/path/to/spherical/autoencoder_final.pt
%env SRUL_PRIOR_CHECKPOINT=/path/to/spherical/cond_rfm_final.pt
!bash scripts/run_cifar10_matched_baselines.sh runs/final_comparison


## VAE prior-geometry comparison

This keeps one KL-regularized VAE fixed and compares standard linear Flow Matching with post-hoc direction-only RFM.


In [ ]:
%env VAE_CHECKPOINT=runs/final_comparison/CIFAR10/seed_0/checkpoints/ldm_autoencoder_final.pt
%env VAE_LATENT_STATS=runs/final_comparison/CIFAR10/seed_0/checkpoints/ldm_latent_stats.pt
%env PREVIOUS_METRICS_CSV=runs/final_comparison/CIFAR10/seed_0/generation_metrics.csv
!bash scripts/run_cifar10_vae_geometry.sh runs/final_comparison


## Medical and face datasets


In [ ]:
# PathMNIST
# !bash scripts/run_pathmnist.sh runs

# CelebA-64
# !bash scripts/run_celeba64.sh runs data/celeba64
